# Pratica — Clase 8: Despliegue con TensorFlow Lite y TensorFlow.js

En esta práctica vas a trabajar con dos aspectos del despliegue de modelos que viste en el notebook de teoría:

1. **Cuantización post-allenamento (post-training quantization):** vas a convertir un modelo entrenado a TensorFlow Lite (TFLite) con y sin cuantización dinámica, y vas a comperr el tamaño de los dos archivos resultantes
2. **Tiempo de inferencia:** vas a medir y comperr cuánto tarda en predecir el modelo Keras original frente al modelo TFLite

Prova completar cada parte marcada con `# TODO` antes de mirar la solución. Si te atascas, la solución está siempre justo debajo, colapsada.

## Preperción: entrenamos un modelo base

Esta parte no es un ejercicio: solo preper el modelo Keras que vas a usar en los dos ejercicios siguientes. Es el mismo tipo de modelo que ya usaste en el notebook de teoría, sobre Fashion MNIST.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import time

print("Versión de TensorFlow:", tf.__version__)

fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

nombres_clases = [
    "Camiseta", "Pantalón", "Jersey", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"
]

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo.fit(x_train, y_train, epochs=5, validation_split=0.1, verbose=1)

perdida_test, precision_test = modelo.evaluate(x_test, y_test, verbose=0)
print(f"Precisión sobre test: {precision_test:.4f}")

---

## Esercizio 1 — Cuantización post-allenamento: comperr tamaños

**Consegna:** convierte el modelo `modelo` a TensorFlow Lite de dos formas distintas:

- Una conversión **sin optimizar**, tal como hiciste en el notebook de teoría
- Una conversión **con cuantización dinámica**, activando `converter.optimizations = [tf.lite.Optimize.DEFAULT]` antes de llamar a `convert()`

Guarda cada versión en un archivo `.tflite` distinto, comprueba el tamaño de cada uno con `os.path.getsize` y calcula cuántas veces más pequeño es el modelo cuantizado.

**Suggerimenti:**
- La cuantización dinámica convierte los pesos de números decimales (float32) a una representación de menor precisión, sin necesidad de un conjunto de datos representativo adicional
- `os.path.getsize(ruta)` devuelve el tamaño del archivo en bytes; divide entre 1024 per pasarlo a kilobytes
- La API es la misma que ya conoces: `tf.lite.TFLiteConverter.from_keras_model(modelo)`

In [ ]:
# Conversión SIN cuantizar (igual que en el notebook de teoría)
converter_normal = tf.lite.TFLiteConverter.from_keras_model(modelo)
tflite_normal = converter_normal.convert()

ruta_normal = "modelo_sin_cuantizar.tflite"
with open(ruta_normal, "wb") as f:
    f.write(tflite_normal)

# TODO: crea un nuevo conversor a partir de `modelo`
# TODO: activa la cuantización dinámica con converter.optimizations = [tf.lite.Optimize.DEFAULT]
# TODO: llama a convert() y guarda el resultado en un archivo "modelo_cuantizado.tflite"

ruta_cuantizado = "modelo_cuantizado.tflite"
# converter_cuantizado = ...
# converter_cuantizado.optimizations = ...
# tflite_cuantizado = ...
# with open(ruta_cuantizado, "wb") as f:
#     f.write(tflite_cuantizado)

In [ ]:
# TODO: calcula el tamaño en KB de ambos archivos con os.path.getsize
# TODO: imprime los dos tamaños y cuántas veces más pequeño es el modelo cuantizado

# tamano_normal_kb = ...
# tamano_cuantizado_kb = ...
# print(f"Sin cuantizar:  {tamano_normal_kb:.1f} KB")
# print(f"Cuantizado:     {tamano_cuantizado_kb:.1f} KB")
# print(f"Reducción: {tamano_normal_kb / tamano_cuantizado_kb:.2f}x")

<details>
<summary>💡 Vedi la soluzione</summary>

```python
# Conversión SIN cuantizar
converter_normal = tf.lite.TFLiteConverter.from_keras_model(modelo)
tflite_normal = converter_normal.convert()

ruta_normal = "modelo_sin_cuantizar.tflite"
with open(ruta_normal, "wb") as f:
    f.write(tflite_normal)

# Conversión CON cuantización dinámica
converter_cuantizado = tf.lite.TFLiteConverter.from_keras_model(modelo)
converter_cuantizado.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_cuantizado = converter_cuantizado.convert()

ruta_cuantizado = "modelo_cuantizado.tflite"
with open(ruta_cuantizado, "wb") as f:
    f.write(tflite_cuantizado)

# Comparamos tamaños
tamano_normal_kb = os.path.getsize(ruta_normal) / 1024
tamano_cuantizado_kb = os.path.getsize(ruta_cuantizado) / 1024

print(f"Sin cuantizar:  {tamano_normal_kb:.1f} KB")
print(f"Cuantizado:     {tamano_cuantizado_kb:.1f} KB")
print(f"Reducción: {tamano_normal_kb / tamano_cuantizado_kb:.2f}x")

# La cuantización dinámica suele reducir el tamaño del archivo a alrededor
# de una cuarta parte, porque convierte los pesos de float32 (32 bits)
# a una representación de 8 bits. La arquitectura del modelo no cambia,
# solo la precisión numérica con la que se guardan sus parámetros.
```
</details>

---

## Esercizio 2 — Comperr el tiempo de inferencia: Keras frente a TFLite

**Consegna:** usando el modelo `modelo` (Keras) y el archivo `modelo_sin_cuantizar.tflite` que creaste en el ejercicio anterior, mide y comper el tiempo que tarda cada uno en classificare el mismo conjunto de 200 imágenes de test. Después, calcula cuántas veces más rápido (o más lento) es uno respecto al otro.

**Suggerimenti:**
- Usa `time.time()` antes y después de cada bloque de predicciones per medir la duración
- El modelo Keras puede predecir el lote completo de una vez con `modelo.predict(lote, verbose=0)`
- El intérprete de TFLite predice una imagen cada vez: necesitas un bucle que llame a `set_tensor`, `invoke` y `get_tensor` per cada imagen
- Recuerda cargar el intérprete con `tf.lite.Interpreter(model_path=...)` y llamar a `allocate_tensors()` antes de usarlo

In [ ]:
num_muestras = 200
lote_prueba = x_test[:num_muestras].astype("float32")

# TODO: mide el tiempo que tarda el modelo Keras en predecir todo el lote de una vez
# inicio = time.time()
# modelo.predict(...)
# tiempo_keras = ...

# TODO: carga el intérprete de TFLite a partir de "modelo_sin_cuantizar.tflite"
# TODO: obtén los detalles de entrada y salida con get_input_details() / get_output_details()

# TODO: mide el tiempo que tarda el intérprete en predecir las 200 imágenes,
#       una por una, dentro de un bucle for
# inicio = time.time()
# for i in range(num_muestras):
#     ...
# tiempo_tflite = ...

In [ ]:
# TODO: imprime ambos tiempos y calcula la relación entre ellos
# print(f"Tiempo Keras:  {tiempo_keras:.4f} s")
# print(f"Tiempo TFLite: {tiempo_tflite:.4f} s")

<details>
<summary>💡 Vedi la soluzione</summary>

```python
num_muestras = 200
lote_prueba = x_test[:num_muestras].astype("float32")

# Tiempo del modelo Keras, prediciendo el lote completo de una sola vez
inicio = time.time()
modelo.predict(lote_prueba, verbose=0)
tiempo_keras = time.time() - inicio

# Cargamos el intérprete de TFLite
interpreter = tf.lite.Interpreter(model_path="modelo_sin_cuantizar.tflite")
interpreter.allocate_tensors()
detalles_entrada = interpreter.get_input_details()
detalles_salida = interpreter.get_output_details()

# Tiempo del modelo TFLite, prediciendo imagen a imagen (así se usa en un dispositivo real)
inicio = time.time()
for i in range(num_muestras):
    interpreter.set_tensor(detalles_entrada[0]["index"], lote_prueba[i:i+1])
    interpreter.invoke()
    _ = interpreter.get_tensor(detalles_salida[0]["index"])
tiempo_tflite = time.time() - inicio

print(f"Tiempo Keras:  {tiempo_keras:.4f} s")
print(f"Tiempo TFLite: {tiempo_tflite:.4f} s")

if tiempo_tflite < tiempo_keras:
    print(f"TFLite fue {tiempo_keras / tiempo_tflite:.2f}x más rápido")
else:
    print(f"Keras fue {tiempo_tflite / tiempo_keras:.2f}x más rápido")

# En una CPU de escritorio o de Colab, la diferencia puede no ser muy grande
# (a veces Keras predice más rápido porque procesa el lote entero de golpe,
# aprovechando mejor la vectorización, mientras que aquí forzamos a TFLite
# a predecir imagen por imagen, que es como se usa en una aplicación real).
# La ventaja de TFLite no está solo en la velocidad en un ordenador de
# escritorio, sino en que es la única opción viable dentro de un móvil o
# un dispositivo embebido, y en que ocupa mucho menos espacio en disco.
```
</details>

## Per reflexionar

Si repites el ejercicio 2 usando `modelo_cuantizado.tflite` en lugar de `modelo_sin_cuantizar.tflite`, ¿esperas que el tiempo de inferencia cambie mucho? Piensa en la diferencia entre el tamaño del archivo (que sí se reduce claramente con la cuantización) y la velocidad de cálculo (que depende también de si el hardware donde ejecutas el modelo aprovecha bien los enteros de 8 bits). No hace falta que lo escribas en ningún sitio: es una pregunta per que la lleves contigo a la clase 9, cuando hables de GPU y TPU (Tensor Processing Unit o Unidad de Procesamiento Tensorial).